# Day 3 — Model Training & Explainability

**Project:** Adaptive Defect Prediction Engine  
**Dataset:** PROMISE KC1 (2,109 modules, ~25% buggy)  
**Model:** XGBoost with StratifiedKFold CV + SHAP explainability  

This notebook evaluates OOF predictions from `train.py` and explores SHAP explanations.  
Run `scripts/day3_run.py` first to generate all artifacts.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import xgboost as xgb
import shap
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    roc_curve, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report,
)

# Paths
OOF_PATH   = ROOT / 'data/processed/oof_predictions.csv'
META_PATH  = ROOT / 'models/model_meta.json'
MODEL_PATH = ROOT / 'models/xgboost_defect_predictor.json'
FM_PATH    = ROOT / 'data/processed/feature_matrix.csv'
SHAP_PATH  = ROOT / 'data/processed/shap_values.csv'

print('Paths configured')

## 1. Load Artifacts

In [ ]:
oof_df = pd.read_csv(OOF_PATH)
with open(META_PATH) as f:
    meta = json.load(f)

best_threshold = meta['best_threshold']
feature_names  = meta['feature_names']

y_true = oof_df['is_buggy'].values
y_prob = oof_df['oof_prob'].values
y_pred = oof_df['oof_pred'].values

print(f'OOF shape: {oof_df.shape}')
print(f'Best threshold (avg from CV): {best_threshold:.3f}')
print(f'CV Mean AUC: {meta["cv_mean_auc"]:.4f}')
print(f'CV Mean F1:  {meta["cv_mean_f1"]:.4f}')
oof_df.head()

## 2. ROC & Precision-Recall Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# --- ROC ---
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = roc_auc_score(y_true, y_prob)
ax1.plot(fpr, tpr, lw=2, color='steelblue', label=f'AUC = {roc_auc:.4f}')
ax1.plot([0,1],[0,1],'k--',lw=1)
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.set_title('ROC Curve (OOF)')
ax1.legend(); ax1.grid(alpha=0.3)

# --- PR ---
prec, rec, _ = precision_recall_curve(y_true, y_prob)
ap = average_precision_score(y_true, y_prob)
ax2.plot(rec, prec, lw=2, color='darkorange', label=f'AP = {ap:.4f}')
ax2.axhline(y_true.mean(), color='gray', ls='--', label=f'Baseline={y_true.mean():.2%}')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve (OOF)')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('KC1 Defect Predictor — OOF Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'AUC-ROC: {roc_auc:.4f}  |  Avg Precision: {ap:.4f}')

## 3. Threshold Sensitivity Analysis

In [ ]:
thresholds = np.arange(0.05, 0.95, 0.01)
f1s, precs, recs = [], [], []

for thr in thresholds:
    preds = (y_prob >= thr).astype(int)
    f1s.append(f1_score(y_true, preds, zero_division=0))
    precs.append(precision_score(y_true, preds, zero_division=0))
    recs.append(recall_score(y_true, preds, zero_division=0))

best_f1_idx = np.argmax(f1s)
best_thr_from_oof = thresholds[best_f1_idx]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, f1s,   label='F1',        color='steelblue', lw=2)
ax.plot(thresholds, precs, label='Precision',  color='darkorange', lw=2)
ax.plot(thresholds, recs,  label='Recall',     color='green', lw=2)
ax.axvline(best_threshold, color='red',   ls='--', lw=1.5, label=f'CV threshold={best_threshold:.2f}')
ax.axvline(best_thr_from_oof, color='purple', ls=':', lw=1.5, label=f'OOF best F1 threshold={best_thr_from_oof:.2f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold Sensitivity: F1 / Precision / Recall')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'OOF best F1={max(f1s):.4f} at threshold={best_thr_from_oof:.2f}')

## 4. Confusion Matrix at Best Threshold

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)
labels = ['Clean', 'Buggy']
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(labels); ax.set_yticklabels(labels)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix @ threshold={best_threshold:.2f}')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black',
                fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=['Clean','Buggy']))

## 5. SHAP Summary Beeswarm

In [ ]:
# Load the feature matrix (aligned to training features)
df_raw = pd.read_csv(FM_PATH)
nan_frac = df_raw.isnull().mean()
drop_cols = nan_frac[nan_frac > 0.50].index.tolist()
df_raw = df_raw.drop(columns=drop_cols, errors='ignore')
for col in ['file_path', 'Unnamed: 0', 'is_buggy']:
    if col in df_raw.columns:
        df_raw = df_raw.drop(columns=[col])
if 'has_process_data' not in df_raw.columns:
    df_raw['has_process_data'] = 0

X_full = df_raw[feature_names]

# Load SHAP values
shap_df = pd.read_csv(SHAP_PATH)
if 'file_path' in shap_df.columns:
    shap_df = shap_df.drop(columns=['file_path'])
shap_vals = shap_df[feature_names].values

print(f'SHAP matrix: {shap_vals.shape}')

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_vals, X_full, plot_type='dot', show=False, max_display=20)
plt.title('SHAP Beeswarm — KC1 Defect Predictor', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Top 10 Most At-Risk Modules with SHAP Explanations

In [ ]:
# Mean |SHAP| global importance
mean_abs_shap = pd.Series(
    np.abs(shap_vals).mean(axis=0),
    index=feature_names,
).sort_values(ascending=False)

print('Top 10 features by Mean |SHAP|:')
print(mean_abs_shap.head(10).to_string())

fig, ax = plt.subplots(figsize=(9, 5))
mean_abs_shap.head(15).sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Global Feature Importance (Mean |SHAP|) — Top 15')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 most at-risk modules (highest predicted probability)
oof_df_full = oof_df.copy()

# Attach top SHAP features per module
shap_df_indexed = pd.read_csv(SHAP_PATH)
top3_feats = mean_abs_shap.head(3).index.tolist()

if 'file_path' in shap_df_indexed.columns:
    shap_top = shap_df_indexed[['file_path'] + top3_feats]
    top10 = (
        oof_df_full
        .sort_values('oof_prob', ascending=False)
        .head(10)
        .merge(shap_top, on='file_path', how='left')
    )
else:
    top10 = oof_df_full.sort_values('oof_prob', ascending=False).head(10)

print('Top 10 Most At-Risk Modules:')
display(top10[['file_path', 'is_buggy', 'oof_prob', 'oof_pred'] + top3_feats])

## 7. Day 3 Summary

### What was built
- **`src/models/train.py`** — StratifiedKFold XGBoost trainer with threshold tuning and MLflow tracking  
- **`src/models/evaluate.py`** — Full evaluation suite (ROC, PR, calibration, confusion matrix)  
- **`src/explainability/shap_explainer.py`** — TreeExplainer with global/local SHAP plots  
- **`scripts/day3_run.py`** — Master orchestration script  

### Key findings
- **Data alignment**: KC1 is a C codebase; process + AST features are 100% NaN and were correctly excluded. XGBoost trains on 21 KC1 static features + `has_process_data` flag (=0).
- **Multicollinearity**: ~30% of Halstead feature pairs have |r| ≥ 0.90. XGBoost handles this via importance redistribution, but explainability is diluted across the cluster. Run with `--drop-halstead` if AUC < 0.70.
- **Expected AUC range**: 0.70–0.78 on KC1 static-only features (literature benchmark).
- **Top SHAP features**: Halstead effort/time/intelligence and design complexity dominate — consistent with Day 2 mutual information findings.

### Day 4 plan
- Graph Neural Network on file-level dependency graph (imports as edges)
- Hybrid fusion: XGBoost probability + GNN embedding
- Add real Flask git history for process feature coverage

In [ ]:
# Final metrics summary
final_preds = (y_prob >= best_threshold).astype(int)

summary = {
    'OOF AUC-ROC':            round(roc_auc_score(y_true, y_prob), 4),
    'OOF Avg Precision':      round(average_precision_score(y_true, y_prob), 4),
    'OOF F1 @ best threshold': round(f1_score(y_true, final_preds, zero_division=0), 4),
    'OOF Precision':          round(precision_score(y_true, final_preds, zero_division=0), 4),
    'OOF Recall':             round(recall_score(y_true, final_preds, zero_division=0), 4),
    'Best threshold':         round(best_threshold, 3),
    'CV Mean AUC':            round(meta['cv_mean_auc'], 4),
    'CV Mean F1':             round(meta['cv_mean_f1'], 4),
    'Top SHAP feature':       mean_abs_shap.index[0],
}

print('\n=== Day 3 Final Summary ===')
for k, v in summary.items():
    print(f'  {k:<30} {v}')